In [1]:
# PREDICTION OF HOUSE PRICES

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
housing = pd.read_csv("housing.csv")

In [3]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)
 
split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].drop("income_cat", axis = 1)
    strat_test_set = housing.loc[test_index].drop("income_cat", axis = 1)

In [4]:
housing = strat_train_set.copy()

In [5]:
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis = 1)

In [6]:
num_attribs = housing.drop("ocean_proximity", axis = 1).columns.tolist()
cat_attribs = ["ocean_proximity"]

In [7]:
# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", StandardScaler()),
])

In [8]:
# Categorical pipeline
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown = "ignore"))
])

In [9]:
# Full pipeline
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

In [20]:
housing_prepared = full_pipeline.fit_transform(housing)
housing_prepared.shape

(16512, 13)

In [35]:
from sklearn.linear_model import LinearRegression 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.model_selection import cross_val_score

In [28]:
linreg = LinearRegression()
linreg.fit(housing_prepared , housing_labels)
prediction_lin = linreg.predict(housing_prepared)
lim_rmse = rmse(housing_labels , prediction_lin)

In [29]:
lim_rmse

69050.56219504568

In [30]:
decreg = DecisionTreeRegressor()
decreg.fit(housing_prepared , housing_labels)
prediction_dec = decreg.predict(housing_prepared)
dec_rmse = rmse(housing_labels , prediction_dec)

In [31]:
dec_rmse # why the error is zero? ...its because the model is overfitted !

0.0

In [37]:
#Cross Validating the Model for Overfitting of Data
dec_rmses = -cross_val_score(
    decreg,
    housing_prepared,
    housing_labels,
    scoring = "neg_root_mean_squared_error",
    cv = 10
)

In [41]:
pd.Series(dec_rmses).describe() #this confirms that model was overfitted as mean is large

count       10.000000
mean     68744.749535
std       2198.277896
min      64462.834536
25%      67441.573911
50%      68789.768745
75%      70185.031322
max      71936.039166
dtype: float64

In [32]:
ramreg = RandomForestRegressor()
ramreg.fit(housing_prepared , housing_labels)
prediction_ram = ramreg.predict(housing_prepared)
ram_rmse = rmse(housing_labels , prediction_ram)

In [34]:
ram_rmse # we will go with the Random Forest Pipeline

18367.711710495998

In [44]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.model_selection import cross_val_score

In [45]:
MODEL_FILE = "model.pkl"
PIPELINE_FILE = "pipeline.pkl"

In [46]:
def build_pipeline(num_attributes , cat_attributes):
    # Numerical pipeline
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipeline = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs)
    ])
    return full_pipeline

In [51]:
if not os.path.exists(MODEL_FILE):
    housing = pd.read_csv("housing.csv")
    housing["income_cat"] = pd.cut(housing["median_income"] , bins = [0.0 , 1.5 , 3.0 , 4.5 , 6.0 , np.inf] , labels = [1 , 2 , 3 , 4 , 5])
    split = StratifiedShuffleSplit(n_splits = 1 , test_size = 0.2 , random_state = 42)

    #getting the train and test(input for the model) sets
    for train_index , test_index in split.split(housing , housing["income_cat"]):
        input_data = housing.loc[test_index].drop("income_cat" , axis = 1).to_csv("input.csv" , index = False)
        housing = housing.loc[train_index].drop("income_cat" , axis = 1)

    housing_labels = housing["median_house_value"].copy()
    housing_features = housing.drop("median_house_value" , axis = 1)

    num_attribs = housing_features.drop("ocean_proximity" , axis = 1).columns.tolist()
    cat_attribs = ["ocean_proximity"]

    #building the overall pipeline here
    pipeline = build_pipeline(num_attribs, cat_attribs)
    housing_prepared = pipeline.fit_transform(housing_features)

    #model fitted
    model = RandomForestRegressor(random_state = 42)
    model.fit(housing_prepared, housing_labels)

    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline, PIPELINE_FILE)
 
    print("Model trained and saved.")

else:
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)
 
    input_data = pd.read_csv("input.csv")
    transformed_input = pipeline.transform(input_data)
    predictions = model.predict(transformed_input)
    input_data["median_house_value"] = predictions
 
    input_data.to_csv("output.csv", index = False)
    print("Inference completed. Results saved to output.csv")

Inference completed. Results saved to output.csv
